# hyperloom tour

One cell per feature. Each `show(...)` call renders inline in the cell output below it (Jupyter is auto-detected — see `hyperloom_bridge.show`'s docstring). Run cells top to bottom.

## Setup

This repo isn't published yet, so point at the local packages directly. If you've installed `hyperloom-core`/`hyperloom-bridge` into your kernel's environment (see the repo README), you can skip the `sys.path` lines.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent.parent  # examples/notebooks/ -> repo root
sys.path.insert(0, str(repo_root / "packages" / "core"))
sys.path.insert(0, str(repo_root / "packages" / "bridge"))

from hyperloom_bridge import show
from hyperloom_core import Graph
import numpy as np

## Plain graph

Nodes + edges, force-directed layout streaming in as it converges.

In [ ]:
g = Graph()
for i in range(20):
    g.add_node(i)
for i in range(19):
    g.add_edge(i, i + 1)
for i in range(0, 19, 4):
    g.add_edge(i, (i + 9) % 20)

show(g, seed=0)

## Directed graph

Arrowheads point from source to target.

In [ ]:
g = Graph()
for i in range(12):
    g.add_node(i)
for i in range(12):
    g.add_edge(i, (i + 1) % 12, directed=True)
for i in range(0, 12, 3):
    g.add_edge(i, (i + 5) % 12, directed=True)

show(g, seed=1)

## Temporal graph

The timeline slider at the bottom (click **All time** to start scrubbing) filters edges by `t_start <= t <= t_end`. The ring backbone has no time bounds, so it's always visible; the cross edges each only exist in a short window.

For a view of the *whole* timeline at once instead of one moment at a time, click **Stack: time** (top-left) — each time-bucket draws as its own separated plane (blue = early, orange = late), with faint threads tracking each node across buckets.

In [ ]:
g = Graph()
for i in range(12):
    g.add_node(i)
for i in range(12):
    g.add_edge(i, (i + 1) % 12)
for i in range(12):
    g.add_edge(i, (i + 5) % 12, t_start=i, t_end=i + 2)

show(g, seed=3)

## Multiplex / multilayer graph

Each layer gets its own edge color and a checkbox in the top-right legend. Edges with no layer (the backbone) always stay visible.

Click **Stack: layers** (top-left) to see each layer as its own separated plane instead of overlaid on one — the clearest way to see "layer" as an actual dimension rather than just a color.

In [ ]:
g = Graph()
for i in range(10):
    g.add_node(i)
g.add_layer("friendship")
g.add_layer("coworker")
for i in range(10):
    g.add_edge(i, (i + 1) % 10)
for i in range(0, 10, 2):
    g.add_edge(i, (i + 3) % 10, layer="friendship")
for i in range(1, 10, 2):
    g.add_edge(i, (i + 4) % 10, layer="coworker")

show(g, seed=5)

## Hypergraph

Each hyperedge (a connector with more than 2 members) renders as a translucent convex-hull polygon wrapping its member nodes, instead of a line.

In [ ]:
g = Graph()
for i in range(12):
    g.add_node(i)
g.add_hyperedge([0, 1, 2, 3])
g.add_hyperedge([3, 4, 5])
g.add_hyperedge([6, 7, 8, 9, 10])
g.add_hyperedge([1, 6, 11])

show(g, seed=2)

## Custom styling

`show()` accepts style kwargs — no JS/CSS required. Colors are `[r, g, b, a]` in 0-1 range. See `hyperloom_bridge.show`'s docstring for the full list (node_color, node_radius_px, edge_color, background_color, arrow_color/length/width/t, hull_padding). Per-attribute styling and node shape aren't supported yet.

In [ ]:
g = Graph()
for i in range(8):
    g.add_node(i)
for i in range(7):
    g.add_edge(i, i + 1)

show(
    g,
    seed=1,
    node_color=[1.0, 0.1, 0.1, 1.0],
    node_radius_px=10,
    background_color=[0.1, 0.1, 0.15, 1.0],
    edge_color=[0.5, 0.5, 0.6, 0.6],
)

## 100K-node scale check

Above ~5000 nodes the layout falls back to edge-driven placement (no pairwise repulsion — see `hyperloom_core.algorithms.layout`'s docstring), so the shape is cruder, but rendering itself stays responsive.

In [ ]:
rng = np.random.default_rng(0)
n = 100_000
g = Graph()
for i in range(n):
    g.add_node(i)
for u, v in rng.integers(0, n, size=(200_000, 2)):
    if u != v:
        g.add_edge(int(u), int(v))

show(g, seed=0, layout_iterations=60)